# TFT Training — Subway Headway Prediction

Temporal Fusion Transformer for predicting **minutes until next train arrival**  
for A/C/E southbound trains on the 8th Avenue line (NYC Subway).

**Environment**: Google Colab with A100 GPU, bf16 mixed precision  
**Data**: Parquet from feature engineering notebook (`bigquery_explorer.ipynb`)  

| Split | Cutoff | Purpose |
|-------|--------|---------|
| Train | < 2025-10-29 | Model fitting |
| Val   | < 2025-12-23 | Early stopping / tuning |
| Test  | < 2026-02-17 | Final holdout evaluation |

**Encoder length**: 30 (from autocorrelation analysis — ACF < 0.05 at lag 29)

In [ ]:
# ── Install dependencies (Colab) ─────────────────────────────────────
%pip install pytorch-forecasting lightning tensorboard --quiet

In [ ]:
# ── Imports & A100 precision config ───────────────────────────────────
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MAE, RMSE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── A100 optimizations ────────────────────────────────────────────────
torch.set_float32_matmul_precision('high')       # enable TF32 on A100
torch.backends.cudnn.benchmark = True             # cuDNN autotuning
PRECISION = 'bf16-mixed'                          # BFloat16 mixed precision

print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'bf16 support: {torch.cuda.is_bf16_supported()}')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────
# Option 1: Upload directly (uncomment)
# from google.colab import files
# uploaded = files.upload()  # upload tft_training_data.parquet
# DATA_PATH = 'tft_training_data.parquet'

# Option 2: Google Drive (uncomment)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/path/to/tft_training_data.parquet'

# Option 3: Local (for testing outside Colab)
DATA_PATH = '../local_artifacts/processed_data/tft_training_data.parquet'

data = pd.read_parquet(DATA_PATH)
print(f'Loaded {len(data):,} rows × {data.shape[1]} columns')
print(f'Columns: {list(data.columns)}')
print(f'\nTime range: {data["arrival_time"].min()} → {data["arrival_time"].max()}')
print(f'Groups: {data["group_id"].nunique()}')
data.head()

## Step 1 — Train / Validation / Test Splits

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────
ENCODER_LENGTH  = 25            # lookback (ACF < 0.10 at lag 23, PACF ≈ 0 by lag 15)
PREDICTION_LENGTH = 1      # single-step prediction
BATCH_SIZE      = 256      # smaller batches → more gradient updates, better regularization
MAX_EPOCHS      = 50
LEARNING_RATE   = 1e-3
HIDDEN_SIZE     = 128
ATTENTION_HEADS = 4
DROPOUT         = 0.1
GRADIENT_CLIP   = 0.1
NUM_WORKERS     = 8

In [ ]:
# ── Temporal split cutoffs (UTC) ─────────────────────────────────────
train_cutoff = pd.Timestamp('2025-10-29', tz='UTC')
val_cutoff   = pd.Timestamp('2025-12-23', tz='UTC')
test_cutoff  = pd.Timestamp('2026-02-17', tz='UTC')

# Label each row
data['split'] = 'test'
data.loc[data['arrival_time'] < val_cutoff,   'split'] = 'val'
data.loc[data['arrival_time'] < train_cutoff,  'split'] = 'train'

split_counts = data['split'].value_counts()
print('Split sizes:')
for s in ['train', 'val', 'test']:
    n = split_counts.get(s, 0)
    pct = n / len(data) * 100
    print(f'  {s:5s}: {n:>10,} rows  ({pct:.1f}%)')

print(f'\nTrain period: {data.loc[data.split=="train","arrival_time"].min()} → {data.loc[data.split=="train","arrival_time"].max()}')
print(f'Val period:   {data.loc[data.split=="val","arrival_time"].min()} → {data.loc[data.split=="val","arrival_time"].max()}')
print(f'Test period:  {data.loc[data.split=="test","arrival_time"].min()} → {data.loc[data.split=="test","arrival_time"].max()}')

In [ ]:
# ── Prepare model-ready DataFrame ────────────────────────────────────
# Drop columns the model shouldn't see; keep arrival_time only for splitting
MODEL_COLS = [
    'time_idx', 'group_id', 'route_id', 'stop_id', 'station_order',
    'minutes_until_next_train',
    'hour', 'day_of_week', 'month',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'is_weekend', 'is_holiday',
    'rolling_mean_3', 'rolling_std_3',
    'rolling_mean_5', 'rolling_std_5',
    'rolling_mean_10', 'rolling_max_10',
    'empirical_median',
    'headway_deviation_ratio', 'headway_deviation_signed',
    'deviation_trend', 'deviation_streak',
    'time_since_upstream_1', 'time_since_upstream_2',
    'upstream_headway_1', 'upstream_headway_2', 'upstream_headway_3',
    'upstream_deviation_ratio',
    'last_train_travel_time_1', 'travel_time_deviation_1', 'last_train_travel_time_2',
    'any_route_headway_upstream', 'any_route_time_since_upstream', 'cross_route_trains_last_10min',
]

# ── Build split DataFrames with encoder context ──────────────────────
# For val/test: prepend ENCODER_LENGTH rows of context per group so the
# encoder has history for the first prediction window.

def build_split_df(data_full, split_start, split_end, encoder_length):
    """Return rows for a split period + encoder context from just before it."""
    period = data_full[
        (data_full['arrival_time'] >= split_start) &
        (data_full['arrival_time'] < split_end)
    ]
    frames = []
    for gid in period['group_id'].unique():
        grp_all = data_full[data_full['group_id'] == gid]
        grp_period = period[period['group_id'] == gid]
        min_tidx = grp_period['time_idx'].min()
        context_start = max(0, min_tidx - encoder_length)
        mask = (
            (grp_all['time_idx'] >= context_start) &
            (grp_all['arrival_time'] < split_end)
        )
        frames.append(grp_all[mask])
    return pd.concat(frames).sort_values(['group_id', 'time_idx']).reset_index(drop=True)

# Training: all data before train_cutoff
train_df = data[data['arrival_time'] < train_cutoff][MODEL_COLS].copy()

# Validation: val period + encoder context from late training
val_df = build_split_df(data, train_cutoff, val_cutoff, ENCODER_LENGTH)[MODEL_COLS].copy()

# Test: test period + encoder context from late validation
test_df = build_split_df(data, val_cutoff, test_cutoff, ENCODER_LENGTH)[MODEL_COLS].copy()

# ── Scale continuous features (fit on train, transform all) ──────────
SCALE_COLS = [
    'station_order',
    'rolling_mean_3', 'rolling_std_3',
    'rolling_mean_5', 'rolling_std_5',
    'rolling_mean_10', 'rolling_max_10',
    'empirical_median',
    'upstream_headway_1', 'upstream_headway_2', 'upstream_headway_3',
    'any_route_headway_upstream', 'any_route_time_since_upstream', 'cross_route_trains_last_10min',
]

scaler = StandardScaler()
train_df[SCALE_COLS] = scaler.fit_transform(train_df[SCALE_COLS])
val_df[SCALE_COLS]   = scaler.transform(val_df[SCALE_COLS])
test_df[SCALE_COLS]  = scaler.transform(test_df[SCALE_COLS])

print('Scaled features (train stats):')
for col, mu, std in zip(SCALE_COLS, scaler.mean_, scaler.scale_):
    print(f'  {col:20s}  mean={mu:.3f}  std={std:.3f}')

print(f'\ntrain_df: {len(train_df):>10,} rows')

print(f'val_df:   {len(val_df):>10,} rows  (incl. {ENCODER_LENGTH}-step context per group)')
print(f'test_df:  {len(test_df):>10,} rows  (incl. {ENCODER_LENGTH}-step context per group)')

In [ ]:
# ── TimeSeriesDataSet: training ──────────────────────────────────────
training_ds = TimeSeriesDataSet(
    train_df,
    time_idx='time_idx',
    target='minutes_until_next_train',
    group_ids=['group_id'],

    max_encoder_length=ENCODER_LENGTH,
    min_encoder_length=ENCODER_LENGTH // 2,
    max_prediction_length=PREDICTION_LENGTH,
    min_prediction_length=1,

    static_categoricals=['route_id', 'stop_id'],
    static_reals=['station_order'],

    time_varying_known_reals=[
        'hour_sin', 'hour_cos',
        'dow_sin', 'dow_cos',
        'month_sin', 'month_cos',
        'is_weekend', 'is_holiday',
        'empirical_median',
    ],

    time_varying_unknown_reals=[
        'rolling_mean_3', 'rolling_std_3',
        'rolling_mean_5', 'rolling_std_5',
        'rolling_mean_10', 'rolling_max_10',
        'headway_deviation_ratio', 'headway_deviation_signed',
        'deviation_trend', 'deviation_streak',
        'time_since_upstream_1', 'time_since_upstream_2',
        'upstream_headway_1', 'upstream_headway_2', 'upstream_headway_3',
        'upstream_deviation_ratio',
        'last_train_travel_time_1', 'travel_time_deviation_1', 'last_train_travel_time_2',
        'any_route_headway_upstream', 'any_route_time_since_upstream', 'cross_route_trains_last_10min',
    ],

    target_normalizer=GroupNormalizer(groups=['group_id']),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# ── Validation & test datasets (inherit params from training) ────────
validation_ds = TimeSeriesDataSet.from_dataset(
    training_ds, val_df,
    stop_randomization=True,
    min_encoder_length=ENCODER_LENGTH,  # enforce full encoder for clean val
)

test_ds = TimeSeriesDataSet.from_dataset(
    training_ds, test_df,
    stop_randomization=True,
    min_encoder_length=ENCODER_LENGTH,
)

print(f'Training samples:   {len(training_ds):>10,}')

print(f'Validation samples: {len(validation_ds):>10,}')
print(f'Test samples:       {len(test_ds):>10,}')

In [ ]:
# ── DataLoaders ──────────────────────────────────────────────────────
train_dl = training_ds.to_dataloader(
    train=True, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    persistent_workers=True,
)
val_dl = validation_ds.to_dataloader(
    train=False, batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS,
    persistent_workers=True,
)
test_dl = test_ds.to_dataloader(
    train=False, batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS,
    persistent_workers=True,
)

# Quick sanity check
x, y = next(iter(train_dl))
print(f'Batch encoder shape: {x["encoder_cont"].shape}')   # (batch, encoder_len, n_cont)
print(f'Batch target shape:  {y[0].shape}')                 # (batch, prediction_len)

## Step 2 — Model Architecture

In [ ]:
# ── TFT model ────────────────────────────────────────────────────────
tft = TemporalFusionTransformer.from_dataset(
    training_ds,
    learning_rate=LEARNING_RATE,
    hidden_size=HIDDEN_SIZE,
    attention_head_size=ATTENTION_HEADS,
    dropout=DROPOUT,
    hidden_continuous_size=HIDDEN_SIZE // 4,
    loss=QuantileLoss(quantiles=[0.1, 0.5, 0.9]),  # p10, p50, p90
    optimizer='adam',
    reduce_on_plateau_patience=3,
    log_interval=-1,                               # disable plot logging (bf16 incompatible with matplotlib)
    log_val_interval=-1,
)

print(f'\nModel parameters: {tft.size() / 1e3:.1f}K')
print(f'\nModel architecture:\n{tft}')

## Step 3 — Training

In [ ]:
# ── Launch TensorBoard (run BEFORE training to monitor live) ────────
%load_ext tensorboard
%tensorboard --logdir lightning_logs

In [ ]:
# ── Trainer ───────────────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min',
    verbose=True,
)

checkpoint = ModelCheckpoint(
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    filename='tft-best-{epoch}-{val_loss:.4f}',
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='gpu',
    devices=1,
    precision=PRECISION,
    gradient_clip_val=GRADIENT_CLIP,
    limit_train_batches=1000,                     # ~256K samples/epoch, 2× more gradient updates
    limit_val_batches=400,                          # cap val steps too
    callbacks=[early_stop, checkpoint, lr_monitor],
    logger=TensorBoardLogger('lightning_logs', name='tft_headway'),
    enable_progress_bar=True,
    log_every_n_steps=50,
)

# ── Train ─────────────────────────────────────────────────────────────
trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)

print(f'\nBest model path: {checkpoint.best_model_path}')
print(f'Best val_loss:   {checkpoint.best_model_score:.4f}')

## Step 4 — Evaluation on Holdout Test Set

In [ ]:
# ── Load best checkpoint ─────────────────────────────────────────────
best_model = TemporalFusionTransformer.load_from_checkpoint(
    checkpoint.best_model_path
)

# ── Generate predictions on test set ─────────────────────────────────
raw_predictions = best_model.predict(
    test_dl, mode='raw', return_x=True, trainer_kwargs=dict(accelerator='gpu')
)

# Point predictions (median = index 1 of [0.1, 0.5, 0.9])
predictions = raw_predictions.output.prediction[:, :, 1].squeeze().cpu()  # (n_samples,)

# Actuals
actuals = torch.cat([y[0] for x, y in iter(test_dl)]).squeeze().cpu()

# ── Metrics ───────────────────────────────────────────────────────────
mae  = torch.abs(predictions - actuals).mean().item()
rmse = torch.sqrt(torch.pow(predictions - actuals, 2).mean()).item()
mape = (torch.abs((actuals - predictions) / actuals.clamp(min=0.1))).mean().item() * 100

print('═' * 50)
print(f'  TEST SET METRICS  ({len(actuals):,} samples)')
print('═' * 50)
print(f'  MAE:  {mae:.3f} minutes')
print(f'  RMSE: {rmse:.3f} minutes')
print(f'  MAPE: {mape:.1f}%')
print('═' * 50)

In [ ]:
# ── Visualizations ───────────────────────────────────────────────────
preds_np = predictions.cpu().numpy()
actuals_np = actuals.cpu().numpy()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Actual vs Predicted scatter
ax = axes[0, 0]
ax.scatter(actuals_np, preds_np, alpha=0.05, s=2, c='steelblue')
lims = [0, 60]
ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')
ax.set_xlabel('Actual (minutes)')
ax.set_ylabel('Predicted (minutes)')
ax.set_title('Actual vs Predicted')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend()

# 2. Residual distribution
ax = axes[0, 1]
residuals = preds_np - actuals_np
ax.hist(residuals, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Residual (predicted - actual) minutes')
ax.set_ylabel('Count')
ax.set_title(f'Residual Distribution (mean={residuals.mean():.2f}, std={residuals.std():.2f})')

# 3. Error by actual headway (binned)
ax = axes[1, 0]
bins = np.arange(0, 65, 5)
bin_idx = np.digitize(actuals_np, bins)
bin_maes = []
bin_centers = []
for i in range(1, len(bins)):
    mask = bin_idx == i
    if mask.sum() > 0:
        bin_maes.append(np.abs(residuals[mask]).mean())
        bin_centers.append((bins[i-1] + bins[i]) / 2)
ax.bar(bin_centers, bin_maes, width=4, color='darkorange', edgecolor='white')
ax.set_xlabel('Actual headway bin (minutes)')
ax.set_ylabel('MAE (minutes)')
ax.set_title('MAE by Headway Range')

# 4. Sample time series: predictions vs actuals for one group
ax = axes[1, 1]
n_show = min(200, len(actuals_np))
ax.plot(range(n_show), actuals_np[:n_show], label='Actual', alpha=0.8, linewidth=1)
ax.plot(range(n_show), preds_np[:n_show], label='Predicted', alpha=0.8, linewidth=1)
ax.set_xlabel('Sample index')
ax.set_ylabel('Minutes until next train')
ax.set_title(f'First {n_show} test predictions vs actuals')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── TFT Interpretability: Variable Importance ────────────────────────
interpretation = best_model.interpret_output(
    raw_predictions.output, reduction='mean'
)

fig = best_model.plot_interpretation(interpretation)
fig.suptitle('TFT Variable Importance & Attention', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Prediction interval coverage ─────────────────────────────────────
# Check how well the 3 quantile predictions calibrate: [p10, p50, p90]
quantile_labels = [0.1, 0.5, 0.9]
all_quantiles = raw_predictions.output.prediction.squeeze()  # (n_samples, 3)

print('Quantile prediction coverage (should ≈ expected):')
for i, q in enumerate(quantile_labels):
    coverage = (actuals.cpu() <= all_quantiles[:, i].cpu()).float().mean().item() * 100
    expected = q * 100
    delta = coverage - expected
    print(f'  q={q:.2f}: expected {expected:5.1f}%  actual {coverage:5.1f}%  (Δ={delta:+.1f}%)')

# 80% prediction interval: p10 to p90
lower = all_quantiles[:, 0].cpu().numpy()  # p10
upper = all_quantiles[:, 2].cpu().numpy()  # p90
in_interval = ((actuals_np >= lower) & (actuals_np <= upper)).mean() * 100
print(f'\n80% prediction interval coverage: {in_interval:.1f}%  (expected ~80%)')